# Imports

In [3]:
import tensorflow as tf
tf.__version__
from tensorflow.keras import layers,models
from tensorflow.keras.applications import MobileNetV2

# Load dataset

In [4]:
dataset_path = "dataset"

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset ="training",
    seed=42,
    image_size=(224,224),
    batch_size=32
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(224, 224),
    batch_size=32
)

Found 6652 files belonging to 6 classes.
Using 5322 files for training.
Found 6652 files belonging to 6 classes.
Using 1330 files for validation.


In [5]:
class_names = train_ds.class_names
print(class_names)

['Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_healthy']


# Normalize Images

In [6]:
normalization_layer = layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x,y:(normalization_layer(x),y))
val_ds = val_ds.map(lambda x,y:(normalization_layer(x),y))

# Load Pretrained Model

We use MobileNetV2 as the feature extractor.

In [7]:
base_model = MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
                         )
base_model.trainable = False

# Build the Model

In [8]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(class_names),activation='softmax')
])

# Compile Model

In [9]:
model.compile(
    optimizer ='adam',
    loss ='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [10]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

Epoch 1/5
167/167 ━━━━━━━━━━━━━━━━━━━━ 61s 348ms/step - accuracy: 0.8437 - loss: 0.4222 - val_accuracy: 0.9218 - val_loss: 0.2087
Epoch 2/5
167/167 ━━━━━━━━━━━━━━━━━━━━ 106s 633ms/step - accuracy: 0.9329 - loss: 0.1892 - val_accuracy: 0.9414 - val_loss: 0.1630
Epoch 3/5
167/167 ━━━━━━━━━━━━━━━━━━━━ 113s 672ms/step - accuracy: 0.9500 - loss: 0.1408 - val_accuracy: 0.9466 - val_loss: 0.1504
Epoch 4/5
167/167 ━━━━━━━━━━━━━━━━━━━━ 93s 552ms/step - accuracy: 0.9634 - loss: 0.1047 - val_accuracy: 0.9511 - val_loss: 0.1378
Epoch 5/5
167/167 ━━━━━━━━━━━━━━━━━━━━ 57s 342ms/step - accuracy: 0.9684 - loss: 0.0910 - val_accuracy: 0.9429 - val_loss: 0.1548


In [11]:
tf.__version__

'2.21.0'

In [12]:

# model = tf.keras.models.load_model("model/plant_model.keras")

In [13]:
model.save("model/plant_model_fixed.keras")

In [15]:
model.export("model/plant_model_serving")

INFO:tensorflow:Assets written to: model/plant_model_serving\assets


INFO:tensorflow:Assets written to: model/plant_model_serving\assets


Saved artifact at 'model/plant_model_serving'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_154')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  2348821848016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2348821848784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2348821848400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2348821851856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2348821852624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2348821848592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2348821847824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2348821852240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2348821848976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2348821850320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2348821851280: Tensor

# test

In [14]:
import numpy as np
from PIL import Image
img = Image.open("test_image.jfif")
img = img.resize((224,224))

img_array = np.array(img)/255.0
img_array = np.expand_dims(img_array, axis=0)

pred = model.predict(img_array)

print(class_names[np.argmax(pred)])

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 697ms/step
Tomato_Late_blight
